In [2]:
!pip install annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 4.2 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp313-cp313-linux_x86_64.whl size=520037 sha256=656d7eca82232aeac04bd4ab1b1b8352896716b01ce9234bd7dd96ca0794a9df
  Stored in directory: /root/.cache/pip/wheels/fc/4e/87/f1f957e7382aa370452cff98276f51686924d5415e449800ed
Successfully built annoy


In [4]:
!pip install annoy faiss-cpu hnswlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 25.3 MB/s  0:00:00
  Created wheel for hnswlib: filename=hnswlib-0.8.0-cp313-cp313-linux_x86_64.whl size=2679799 sha256=6ea24dda079aafa96e7f9c14d956461372ff89d53d713bbf75dee104abb3eb66
  Stored in directory: /root/.cache/pip/wheels/35/04/88/b31765a4b9957705e18065db4657e61fc8da54f50e3ef0b67e
Successfully built hnswlib
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [faiss-cpu]


In [5]:
import numpy as np
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

In [6]:
	import numpy as np
	import time
	from annoy import AnnoyIndex
	import faiss
	import hnswlib

	# ===============================
	# Build 1 million rows with 5 dimensions
	# ===============================
	n_data = 1_000_000   # use 100_000 if memory is constrained
	dim = 5
	X = np.random.random((n_data, dim)).astype(np.float32)

	# Query point
	query = np.random.random((1, dim)).astype(np.float32)
	k = 10

	# ===============================
	# 2. Annoy
	# ===============================
	print("=== Annoy ===")
	ann_index = AnnoyIndex(dim, 'euclidean')

	start = time.time()
	for i in range(n_data):
	    ann_index.add_item(i, X[i])
	ann_index.build(10)  # 10 trees
	build_time = time.time() - start

	start = time.time()
	neighbors = ann_index.get_nns_by_vector(query[0], k, include_distances=True)
	query_time = time.time() - start

	print("Build time:", build_time, "seconds")
	print("Query time:", query_time, "seconds")
	print("Neighbors:", neighbors[0][:5], "...")

	# ===============================
	# 3. FAISS (Flat Index)
	# ===============================
	print("\n=== FAISS (IndexFlatL2) ===")
	faiss_index = faiss.IndexFlatL2(dim)

	start = time.time()
	faiss_index.add(X)
	build_time = time.time() - start

	start = time.time()
	distances, indices = faiss_index.search(query, k)
	query_time = time.time() - start

	print("Build time:", build_time, "seconds")
	print("Query time:", query_time, "seconds")
	print("Neighbors:", indices[0][:5], "...")

	# ===============================
	# 4. HNSW (hnswlib)
	# ===============================
	print("\n=== HNSW (hnswlib) ===")
	hnsw_index = hnswlib.Index(space='l2', dim=dim)

	start = time.time()
	hnsw_index.init_index(max_elements=n_data, ef_construction=200, M=16)
	hnsw_index.add_items(X)
	build_time = time.time() - start

	hnsw_index.set_ef(50)

	start = time.time()
	labels, distances = hnsw_index.knn_query(query, k=k)
	query_time = time.time() - start

	print("Build time:", build_time, "seconds")
	print("Query time:", query_time, "seconds")
	print("Neighbors:", labels[0][:5], "...")

=== Annoy ===
Build time: 23.893384218215942 seconds
Query time: 0.000217437744140625 seconds
Neighbors: [240466, 190019, 380233, 90316, 423337] ...

=== FAISS (IndexFlatL2) ===
Build time: 0.01370859146118164 seconds
Query time: 0.013228416442871094 seconds
Neighbors: [240466 190019 380233  90316 423337] ...

=== HNSW (hnswlib) ===
Build time: 161.8989953994751 seconds
Query time: 0.00018835067749023438 seconds
Neighbors: [240466 190019 380233  90316 423337] ...
